# Notebook de Modelo Dimensional y Validación — Hito 3

**Proyecto:** Business Intelligence — Universidades del Ecuador  
**Equipo:** Equipo 6, FIS-EPN  
**Lote de datos:** `f7c0675f-e871-49c5-aab6-2f9560c25658`  
**Propósito:** Replicar y auditar en Python las transformaciones de Power Query del Hito 3, construir el esquema en constelación (6 dimensiones + 3 tablas de hechos), verificar los totales de control (`ControlModelo` y `ControlDimensiones`) y validar las respuestas a las preguntas de negocio P1, P2 y P3.

## Alcance del Modelo Dimensional
- **Dimensiones conformadas:** `DimTiempo` (2024–2026 + clave `0` «Sin fecha») y `DimInstitucion` (64 IES de `cobertura_extraccion.csv`).
- **Dimensiones específicas:** `DimPerfilPrograma` (P1), `DimCargo` y `DimCondicion` (P2), `DimCategoriaPublicacion` (P3).
- **Tablas de hechos:** `FactOferta`, `FactAutoridades` y `FactPublicaciones`.
- **Privacidad (LOPDP):** No se cargan `nombre_completo`, `unidad_adscripcion`, `titulo`, `resumen` ni URLs.

## Cómo ejecutar
```bash
# Desde la raíz del repositorio (epn-dwh-scraper)
jupyter nbconvert --to notebook --execute notebooks/hito3_modelo_dimensional.ipynb
```

In [ ]:
# ─────────────────────────────────────────────
# 0. CONFIGURACIÓN E IMPORTACIONES
# ─────────────────────────────────────────────
import sys
import os
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

# Determinar raíz del proyecto de forma robusta
current = Path.cwd()
project_root = current
for p in [current] + list(current.parents):
    if (p / "src").exists():
        project_root = p
        break
    elif (p / "epn-dwh-scraper" / "src").exists():
        project_root = p / "epn-dwh-scraper"
        break

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Localizar carpeta de los 4 CSV del lote (busca primero data/raw/hito3, luego hito2/<lote>)
LOTE_ID = "f7c0675f-e871-49c5-aab6-2f9560c25658"
candidatos = [
    project_root / "data" / "raw" / "hito3",
    project_root / "data" / "raw" / "hito2" / LOTE_ID,
    Path.cwd() / "data" / "raw" / "hito3",
]

RUTA_LOTE = None
for c in candidatos:
    if (c / "oferta_raw.csv").exists():
        RUTA_LOTE = c.resolve()
        break

if RUTA_LOTE is None:
    raise FileNotFoundError(
        "No se encontraron los CSV del lote (oferta_raw.csv, autoridades_raw.csv, "
        "noticias_raw.csv, cobertura_extraccion.csv) en data/raw/hito3 ni data/raw/hito2."
    )

print(f"Python: {sys.version.split()[0]}")
print(f"Pandas: {pd.__version__}")
print(f"Numpy:  {np.__version__}")
print(f"Ruta del lote: {RUTA_LOTE}")
print(f"Fecha de ejecución: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Funciones de Limpieza y Transformación (Espejo de Power Query `fn*`)

Estas funciones replican exactamente la lógica de `fnTexto`, `fnCategoria`, `fnTipoRegistro`, `fnGrupoCargo`, `fnFecha` y `fnClaveTiempo` definidas en `Proyecto_Equipo6_Hito3.pbix`.

In [ ]:
# ─────────────────────────────────────────────
# 1. FUNCIONES AUXILIARES (ESPEJO DE POWER QUERY)
# ─────────────────────────────────────────────
VACIOS = {"", "no_determinado", "no determinado", "sin categoria", "sin categoría", "nan", "none", "null"}

MAPA_TIPO_REGISTRO = {
    "enlace_oficial_con_titulo": "Programa con título oficial",
    "enlace_no_clasificado": "Enlace no clasificado",
    "pagina_sin_enlaces_clasificables": "Página sin enlaces clasificables",
}

def fn_leer_csv(nombre_archivo: str) -> pd.DataFrame:
    ruta = RUTA_LOTE / nombre_archivo
    df = pd.read_csv(ruta, dtype=str, encoding="utf-8-sig")
    print(f"✅ Cargado {nombre_archivo}: {len(df)} filas, {len(df.columns)} columnas")
    return df

def fn_texto(valor, por_defecto: str = "No determinado") -> str:
    if pd.isna(valor):
        return por_defecto
    t = str(valor).strip()
    return por_defecto if t == "" else t

def fn_categoria(valor, por_defecto: str = "No determinado") -> str:
    if pd.isna(valor):
        return por_defecto
    t = str(valor).strip()
    if t.lower() in VACIOS:
        return por_defecto
    return t.replace("_", " ").title()

def fn_tipo_registro(valor) -> str:
    if pd.isna(valor):
        return "No determinado"
    t = str(valor).strip().lower()
    return MAPA_TIPO_REGISTRO.get(t, "No determinado")

def fn_grupo_cargo(cargo) -> str:
    if pd.isna(cargo):
        return "No determinado"
    c = str(cargo).strip().lower()
    if c == "":
        return "No determinado"
    if c == "autoridad universitaria":
        return "Autoridad universitaria (sin cargo)"
    if c.startswith("vicerrector"):
        return "Vicerrector"
    if c.startswith("rector"):
        return "Rector"
    if c.startswith("decan"):
        return "Decano"
    if c.startswith("director"):
        return "Director"
    return "Otro"

def fn_fecha(serie: pd.Series) -> pd.Series:
    return pd.to_datetime(serie.astype(str).str.strip().str.slice(0, 10), format="%Y-%m-%d", errors="coerce")

def fn_clave_tiempo(serie_fecha: pd.Series) -> pd.Series:
    return serie_fecha.apply(lambda f: 0 if pd.isna(f) else int(f.year * 10000 + f.month * 100 + f.day)).astype("int64")

## 2. Capa de Staging (`StgOferta`, `StgAutoridades`, `StgPublicaciones`)

Se seleccionan únicamente las columnas analíticas (excluyendo datos personales y ruido), se estandarizan categorías y se derivan las claves de tiempo.

In [ ]:
# ─────────────────────────────────────────────
# 2. CONSTRUCCIÓN DE TABLAS STAGING
# ─────────────────────────────────────────────
# StgOferta
raw_oferta = fn_leer_csv("oferta_raw.csv")
cols_oferta = ["institucion_id", "hash_registro", "fecha_extraccion", "provincia", "canton",
               "tipo_registro", "nivel_formacion", "modalidad", "financiamiento", "estado_programa"]
stg_oferta = raw_oferta.reindex(columns=cols_oferta).copy()
stg_oferta["institucion_id"] = stg_oferta["institucion_id"].apply(lambda x: fn_texto(x, "SIN_IES").upper())
stg_oferta["tipo_registro"] = stg_oferta["tipo_registro"].apply(fn_tipo_registro)
for col in ["nivel_formacion", "modalidad", "financiamiento", "estado_programa"]:
    stg_oferta[col] = stg_oferta[col].apply(fn_categoria)
stg_oferta["fecha_extraccion"] = fn_fecha(stg_oferta["fecha_extraccion"])
stg_oferta["tiempo_extraccion_key"] = fn_clave_tiempo(stg_oferta["fecha_extraccion"])

# StgAutoridades (excluye nombre_completo y unidad_adscripcion)
raw_aut = fn_leer_csv("autoridades_raw.csv")
cols_aut = ["institucion_id", "hash_registro", "fecha_observacion", "provincia", "canton",
            "cargo", "cargo_normalizado", "tipo_cargo", "condicion_designacion"]
stg_autoridades = raw_aut.reindex(columns=cols_aut).copy()
stg_autoridades["institucion_id"] = stg_autoridades["institucion_id"].apply(lambda x: fn_texto(x, "SIN_IES").upper())
stg_autoridades["cargo_normalizado"] = stg_autoridades["cargo_normalizado"].apply(fn_texto)
stg_autoridades["tipo_cargo"] = stg_autoridades["tipo_cargo"].apply(fn_categoria)
stg_autoridades["condicion_designacion"] = stg_autoridades["condicion_designacion"].apply(fn_categoria)
stg_autoridades["cargo_grupo"] = stg_autoridades["cargo"].apply(fn_grupo_cargo)
stg_autoridades["fecha_observacion"] = fn_fecha(stg_autoridades["fecha_observacion"])
stg_autoridades["tiempo_observacion_key"] = fn_clave_tiempo(stg_autoridades["fecha_observacion"])

# StgPublicaciones (excluye titulo, resumen y url)
raw_pub = fn_leer_csv("noticias_raw.csv")
cols_pub = ["institucion_id", "hash_registro", "fecha_publicacion", "provincia", "canton",
            "categoria", "tipo_publicacion"]
stg_publicaciones = raw_pub.reindex(columns=cols_pub).copy()
stg_publicaciones["institucion_id"] = stg_publicaciones["institucion_id"].apply(lambda x: fn_texto(x, "SIN_IES").upper())
stg_publicaciones["categoria"] = stg_publicaciones["categoria"].apply(lambda x: fn_categoria(x, "Sin categoría"))
stg_publicaciones["tipo_publicacion"] = stg_publicaciones["tipo_publicacion"].apply(fn_categoria)
stg_publicaciones["fecha_publicacion"] = fn_fecha(stg_publicaciones["fecha_publicacion"])
stg_publicaciones["tiempo_publicacion_key"] = fn_clave_tiempo(stg_publicaciones["fecha_publicacion"])

print(f"\nResumen Staging -> StgOferta: {len(stg_oferta)} | StgAutoridades: {len(stg_autoridades)} | StgPublicaciones: {len(stg_publicaciones)}")

## 3. Construcción de Dimensiones (`Dim*`)

Se generan las 6 dimensiones del modelo con claves sustitutas enteras (`*_key`):
1. `DimTiempo` (conformada)
2. `DimInstitucion` (conformada, 64 IES)
3. `DimPerfilPrograma`
4. `DimCargo`
5. `DimCondicion`
6. `DimCategoriaPublicacion`

In [ ]:
# ─────────────────────────────────────────────
# 3. CONSTRUCCIÓN DE DIMENSIONES
# ─────────────────────────────────────────────
MESES_ES = {
    1: "Enero", 2: "Febrero", 3: "Marzo", 4: "Abril", 5: "Mayo", 6: "Junio",
    7: "Julio", 8: "Agosto", 9: "Septiembre", 10: "Octubre", 11: "Noviembre", 12: "Diciembre"
}

# 3.1 DimTiempo (2024-01-01 a 2026-12-31 + fila 0 Sin fecha)
fechas = pd.date_range("2024-01-01", "2026-12-31", freq="D")
dim_tiempo = pd.DataFrame({"fecha": fechas})
dim_tiempo["tiempo_key"] = (dim_tiempo["fecha"].dt.year * 10000 + dim_tiempo["fecha"].dt.month * 100 + dim_tiempo["fecha"].dt.day).astype("int64")
dim_tiempo["anio"] = dim_tiempo["fecha"].dt.year.astype("Int64")
dim_tiempo["trimestre"] = dim_tiempo["fecha"].dt.quarter.astype("Int64")
dim_tiempo["anio_trimestre"] = dim_tiempo["anio"].astype(str) + "-T" + dim_tiempo["trimestre"].astype(str)
dim_tiempo["mes"] = dim_tiempo["fecha"].dt.month.astype("Int64")
dim_tiempo["mes_nombre"] = dim_tiempo["mes"].map(MESES_ES)

fila_sin_fecha = pd.DataFrame([{
    "tiempo_key": 0, "fecha": pd.NaT, "anio": pd.NA, "trimestre": pd.NA,
    "anio_trimestre": "Sin fecha", "mes": pd.NA, "mes_nombre": "Sin fecha"
}])
dim_tiempo = pd.concat([fila_sin_fecha, dim_tiempo[["tiempo_key", "fecha", "anio", "trimestre", "anio_trimestre", "mes", "mes_nombre"]]], ignore_index=True)

# 3.2 DimInstitucion (64 IES de cobertura_extraccion.csv + geografía)
raw_cob = fn_leer_csv("cobertura_extraccion.csv")
base_ies = raw_cob.reindex(columns=["institucion_id", "nombre_ies"]).copy()
base_ies["institucion_id"] = base_ies["institucion_id"].apply(lambda x: fn_texto(x, "SIN_IES").upper())
base_ies["nombre_ies"] = base_ies["nombre_ies"].apply(fn_texto)
base_ies = base_ies.drop_duplicates(subset=["institucion_id"])

geo = pd.concat([
    stg_oferta[["institucion_id", "provincia", "canton"]],
    stg_autoridades[["institucion_id", "provincia", "canton"]],
    stg_publicaciones[["institucion_id", "provincia", "canton"]],
], ignore_index=True).drop_duplicates(subset=["institucion_id"])

dim_institucion = base_ies.merge(geo, on="institucion_id", how="left")
dim_institucion["provincia"] = dim_institucion["provincia"].apply(fn_texto)
dim_institucion["canton"] = dim_institucion["canton"].apply(fn_texto)
dim_institucion = dim_institucion.sort_values("institucion_id").reset_index(drop=True)
dim_institucion.insert(0, "institucion_key", np.arange(1, len(dim_institucion) + 1, dtype="int64"))

# 3.3 DimPerfilPrograma
cols_perfil = ["tipo_registro", "nivel_formacion", "modalidad", "financiamiento", "estado_programa"]
dim_perfil_programa = stg_oferta[cols_perfil].drop_duplicates().reset_index(drop=True)
dim_perfil_programa.insert(0, "perfil_programa_key", np.arange(1, len(dim_perfil_programa) + 1, dtype="int64"))

# 3.4 DimCargo
cols_cargo = ["cargo_grupo", "cargo_normalizado", "tipo_cargo"]
dim_cargo = stg_autoridades[cols_cargo].drop_duplicates().reset_index(drop=True)
dim_cargo.insert(0, "cargo_key", np.arange(1, len(dim_cargo) + 1, dtype="int64"))

# 3.5 DimCondicion (dominio cerrado de 4 miembros)
valores_cond = sorted(set(["Titular", "Encargado", "Subrogante", "No determinado"]) | set(stg_autoridades["condicion_designacion"].unique()))
dim_condicion = pd.DataFrame({"condicion_designacion": valores_cond})
dim_condicion.insert(0, "condicion_key", np.arange(1, len(dim_condicion) + 1, dtype="int64"))

# 3.6 DimCategoriaPublicacion
cols_cat = ["categoria", "tipo_publicacion"]
dim_categoria_publicacion = stg_publicaciones[cols_cat].drop_duplicates().reset_index(drop=True)
dim_categoria_publicacion.insert(0, "categoria_key", np.arange(1, len(dim_categoria_publicacion) + 1, dtype="int64"))

print("✅ 6 Dimensiones construidas correctamente.")

## 4. Construcción de Tablas de Hechos (`Fact*`)

Cada hecho contiene únicamente claves foráneas (`*_key`), el identificador de linaje (`hash_registro`) y medidas aditivas.

In [ ]:
# ─────────────────────────────────────────────
# 4. CONSTRUCCIÓN DE TABLAS DE HECHOS
# ─────────────────────────────────────────────
# 4.1 FactOferta
fact_oferta = (
    stg_oferta
    .merge(dim_institucion[["institucion_key", "institucion_id"]], on="institucion_id", how="left")
    .merge(dim_perfil_programa, on=cols_perfil, how="left")
)
fact_oferta["Registros"] = 1
fact_oferta = fact_oferta[["tiempo_extraccion_key", "institucion_key", "perfil_programa_key", "hash_registro", "Registros"]]

# 4.2 FactAutoridades
fact_autoridades = (
    stg_autoridades
    .merge(dim_institucion[["institucion_key", "institucion_id"]], on="institucion_id", how="left")
    .merge(dim_cargo, on=cols_cargo, how="left")
    .merge(dim_condicion, on="condicion_designacion", how="left")
)
fact_autoridades["Autoridades"] = 1
fact_autoridades["Encargadas"] = fact_autoridades["condicion_designacion"].isin(["Encargado", "Subrogante"]).astype("int64")
fact_autoridades = fact_autoridades[["tiempo_observacion_key", "institucion_key", "cargo_key", "condicion_key", "hash_registro", "Autoridades", "Encargadas"]]

# 4.3 FactPublicaciones
fact_publicaciones = (
    stg_publicaciones
    .merge(dim_institucion[["institucion_key", "institucion_id"]], on="institucion_id", how="left")
    .merge(dim_categoria_publicacion, on=cols_cat, how="left")
)
fact_publicaciones["Publicaciones"] = 1
fact_publicaciones = fact_publicaciones[["tiempo_publicacion_key", "institucion_key", "categoria_key", "hash_registro", "Publicaciones"]]

print("✅ 3 Tablas de Hechos construidas correctamente.")

## 5. Auditoría del Modelo (`ControlModelo` y `ControlDimensiones`)

Verificación automática de paridad de filas (origen = hecho), integridad referencial (`0` claves sin pareja), rango temporal (`0` fechas fuera de rango) y unicidad de claves en todas las dimensiones.

In [ ]:
# ─────────────────────────────────────────────
# 5. TABLAS DE CONTROL (EQUIVALENTES A POWER BI)
# ─────────────────────────────────────────────
claves_tiempo_validas = set(dim_tiempo["tiempo_key"])

def evaluar_hecho(nombre, stg, fact, claves_fk, col_tiempo):
    sin_pareja = sum(int(fact[c].isna().sum()) for c in claves_fk)
    fuera_rango = int((~fact[col_tiempo].isin(claves_tiempo_validas)).sum())
    return {
        "tabla": nombre,
        "filas_origen": len(stg),
        "filas_hecho": len(fact),
        "coinciden": len(stg) == len(fact),
        "claves_sin_pareja": sin_pareja,
        "fechas_fuera_de_rango": fuera_rango,
    }

control_modelo = pd.DataFrame([
    evaluar_hecho("FactOferta", stg_oferta, fact_oferta, ["institucion_key", "perfil_programa_key"], "tiempo_extraccion_key"),
    evaluar_hecho("FactAutoridades", stg_autoridades, fact_autoridades, ["institucion_key", "cargo_key", "condicion_key"], "tiempo_observacion_key"),
    evaluar_hecho("FactPublicaciones", stg_publicaciones, fact_publicaciones, ["institucion_key", "categoria_key"], "tiempo_publicacion_key"),
])

control_dimensiones = pd.DataFrame([
    {"dimension": nombre, "filas": len(df), "claves_distintas": df[pk].nunique(), "clave_unica": len(df) == df[pk].nunique()}
    for nombre, df, pk in [
        ("DimTiempo", dim_tiempo, "tiempo_key"),
        ("DimInstitucion", dim_institucion, "institucion_key"),
        ("DimPerfilPrograma", dim_perfil_programa, "perfil_programa_key"),
        ("DimCargo", dim_cargo, "cargo_key"),
        ("DimCondicion", dim_condicion, "condicion_key"),
        ("DimCategoriaPublicacion", dim_categoria_publicacion, "categoria_key"),
    ]
])

print("CONTROL DEL MODELO (Hechos)")
print("=" * 65)
display(control_modelo)

print("\nCONTROL DE DIMENSIONES (Unicidad de claves)")
print("=" * 65)
display(control_dimensiones)

# Aserciones estrictas de aceptación
assert control_modelo["coinciden"].all(), "Error: diferencia de filas entre staging y hechos"
assert (control_modelo["claves_sin_pareja"] == 0).all(), "Error: existen claves foráneas sin pareja"
assert (control_modelo["fechas_fuera_de_rango"] == 0).all(), "Error: existen fechas fuera de DimTiempo"
assert control_dimensiones["clave_unica"].all(), "Error: existen claves duplicadas en dimensiones"
print("\n✅ TODAS LAS VALIDACIONES DE INTEGRIDAD PASARON CORRECTAMENTE.")

## 6. Validación de Indicadores DAX y Preguntas de Negocio (P1, P2 y P3)

Se calculan las 5 medidas DAX definidas en Power BI y se replica la **matriz de prueba de P3** (publicaciones por `anio_trimestre` y `categoria` para 2024–2025, total esperado = **232**).

In [ ]:
# ─────────────────────────────────────────────
# 6. MEDIDAS DAX EQUIVALENTES Y PRUEBA P3
# ─────────────────────────────────────────────
# P1: Oferta
oferta_full = fact_oferta.merge(dim_perfil_programa, on="perfil_programa_key", how="left")
registros_oferta = int(oferta_full["Registros"].sum())
programas_oficiales = int(oferta_full.loc[oferta_full["tipo_registro"] == "Programa con título oficial", "Registros"].sum())

# P2: Autoridades
total_autoridades = int(fact_autoridades["Autoridades"].sum())
total_encargadas = int(fact_autoridades["Encargadas"].sum())
pct_encargadas = (total_encargadas / total_autoridades * 100) if total_autoridades else 0.0

# P3: Publicaciones
total_publicaciones = int(fact_publicaciones["Publicaciones"].sum())

kpis = pd.DataFrame([
    {"Pregunta": "P1", "Medida DAX": "Registros de oferta", "Valor": registros_oferta},
    {"Pregunta": "P1", "Medida DAX": "Programas con título oficial", "Valor": programas_oficiales},
    {"Pregunta": "P2", "Medida DAX": "Total Autoridades", "Valor": total_autoridades},
    {"Pregunta": "P2", "Medida DAX": "% Encargadas", "Valor": f"{pct_encargadas:.2f}% ({total_encargadas}/{total_autoridades})"},
    {"Pregunta": "P3", "Medida DAX": "Total Publicaciones", "Valor": total_publicaciones},
])

print("RESUMEN DE MEDIDAS DAX")
print("=" * 55)
display(kpis)

# Matriz de prueba P3 (idéntica a la Vista de Informe de Power BI)
pub_full = (
    fact_publicaciones
    .merge(dim_tiempo[["tiempo_key", "anio", "anio_trimestre"]], left_on="tiempo_publicacion_key", right_on="tiempo_key", how="left")
    .merge(dim_categoria_publicacion[["categoria_key", "categoria"]], on="categoria_key", how="left")
)

pub_24_25 = pub_full[pub_full["anio"].isin([2024, 2025])]
matriz_p3 = pd.pivot_table(
    pub_24_25,
    index="anio_trimestre",
    columns="categoria",
    values="Publicaciones",
    aggfunc="sum",
    fill_value=0,
    margins=True,
    margins_name="Total"
)

print("\nMATRIZ DE PRUEBA P3 — Publicaciones por Trimestre y Categoría (2024–2025)")
print("=" * 75)
display(matriz_p3)

sin_fecha_count = int((fact_publicaciones["tiempo_publicacion_key"] == 0).sum())
print(f"\nVerificación P3: {len(pub_24_25)} fechadas (2024–2025) + {sin_fecha_count} 'Sin fecha' = {len(pub_24_25) + sin_fecha_count} totales.")

In [ ]:
print(f"Notebook Hito 3 ejecutado el: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("Fin de la validación del modelo dimensional (Hito 3).")